[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/annotations/custom-annotations-tool-for-eval-driven-development.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

# **使用自定义标注界面构建评测驱动的开发流程**

本教程介绍如何借助 [Lovable](https://lovable.dev) 为 Phoenix 构建自定义标注界面，并开展实验、评估应用。

自定义标注界面旨在让任何人都能轻松地为追踪提供结构化人工反馈，并将关键信息直接记录到 Phoenix。标注是人工审核时收集反馈的重要手段，有助于持续改进 LLM 应用。

建立反馈闭环和评测流程，可以有效监控并提升系统表现。

本教程需要 [Phoenix Cloud 账号](https://app.arize.com/auth/phoenix/signup) 和 OpenAI API 密钥。

## 安装依赖并配置密钥

In [ ]:
!pip install -qqq arize-phoenix arize-phoenix-otel openinference-instrumentation-openai

In [ ]:
!pip install -qq openai nest_asyncio

In [ ]:
import os
from getpass import getpass

import nest_asyncio

nest_asyncio.apply()

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 采集端地址")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint


if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")

os.environ["OPENAI_API_KEY"] = openai_api_key

# 配置追踪

In [ ]:
from phoenix.otel import register

# 配置 Phoenix 追踪器
tracer_provider = register(
    project_name="my-annotations-app",  # 默认值为 'default'
    auto_instrument=True,
)

# 生成待标注的追踪

先生成一些 Agent 追踪并发送到 Phoenix，再通过标注为特定 Span 添加标签、分数或解释。标注为追踪补充结构化反馈，便于筛选、跟踪和改进 LLM 输出。

In [ ]:
questions = [
    "What is the capital of France?",
    "Who wrote 'Pride and Prejudice'?",
    "What is the boiling point of water in Celsius?",
    "What is the largest planet in our solar system?",
    "Who developed the theory of relativity?",
    "What is the chemical symbol for gold?",
    "In which year did the Apollo 11 mission land on the moon?",
    "What language has the most native speakers worldwide?",
    "Which continent has the most countries?",
    "What is the square root of 144?",
    "What is the largest country in the world by land area?",
    "Why is the sky blue?",
    "Who painted the Mona Lisa?",
    "What is the smallest prime number?",
    "What gas do plants absorb from the atmosphere?",
    "Who was the first President of the United States?",
    "What is the currency of Japan?",
    "How many continents are there on Earth?",
    "What is the tallest mountain in the world?",
    "Who is the author of '1984'?",
]

为展示如何标注不同结果并开展实验，这里通过系统提示词刻意生成一些表现不佳或不合逻辑的追踪。



In [ ]:
from openai import OpenAI

openai_client = OpenAI()

# 系统提示词
system_prompt = """
你是问答助手。对每个用户问题，随机选择 NONSENSE 或 RHYME。如果选择 RHYME，请以押韵的形式正确回答问题。

如果选择 NONSENSE，完全忽略用户问题，不要回答，而是输出不押韵的无意义词语与随机数字。
选择 NONSENSE 时，回答至少包含五个无意义词语，以及五个介于 0 和 9999 之间的随机数字。

不要解释你的选择。
"""

# 遍历数据集并收集 Span
for question in questions:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )

# 打开自定义标注界面

可访问此实现：https://phoenix-trace-annotator.lovable.app/

*注意：此标注界面面向 Phoenix Cloud 演示，未针对大规模追踪工作流优化。*

在 Lovable 中标注追踪的步骤：

1. 输入 Phoenix Cloud 地址、API 密钥和项目名。也可以提供标识符，将标注关联到特定用户。
2. 点击“刷新追踪”（Refresh Traces）。
3. 选择待标注的追踪，点击“发送到 Phoenix”（Send to Phoenix）。
4. 在 Phoenix 中查看新增标注。

运行下面的单元体验此流程：

In [ ]:
from IPython.display import HTML

video_url = (
    "https://storage.googleapis.com/arize-phoenix-assets/assets/videos/custom-annotations-UI.mp4"
)

HTML(f"""
<iframe width="1200" height="700" src="{video_url}"
frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
allowfullscreen></iframe>
""")

# 从已标注的 Span 创建数据集

完成一些 Span 标注后，将它们保存为 Phoenix 数据集。

In [ ]:
import pandas as pd
from phoenix.client import Client
from phoenix.client.types import spans

client = Client()
# 若使用其他标注标准，请替换 "correctness"
query = spans.SpanQuery().where("annotations['correctness']")
spans_df = client.spans.get_spans_dataframe(query=query, project_identifier="my-annotations-app")
dataset = Client().datasets.create_dataset(
    dataframe=spans_df,
    name="annotated-rhymes",
    input_keys=["attributes.input.value"],
    output_keys=["attributes.llm.output_messages"],
)

# 根据标注构建评估器

接下来构建 LLM 评审模板，用来评估实验。此评估器会将不合逻辑的输出判为错误。随着实验中系统提示词的改进，可以观察评测结果的变化。确认已标注追踪数据集上的结果持续改善后，再考虑将修改应用到生产系统。

In [ ]:
RHYME_PROMPT_TEMPLATE = """
检查对话中助手的回答，判断其中是否使用了押韵。

押韵指助手回答在同一行内或不同行之间具有明确的尾韵，且应贯穿整个回答。
回答中不应出现无关短语或数字。
除判断是否押韵外，还要判断押韵自然优质还是生硬牵强。
以上是判断押韵写作质量的标准。

如果助手的所有回答都不押韵，则判为未押韵。

[BEGIN DATA]
    ************
    [Question]: {question}
    ************
    [Response]: {answer}
    [END DATA]


只返回一个英文单词 "correct" 或 "incorrect"，不得包含其他文本或字符。

"correct" 表示回答包含优质的押韵。

"incorrect" 表示回答未押韵。
"""

# 实验示例一：更换模型

In [ ]:
import json

from phoenix.client.__generated__.v1 import DatasetExample as Example
from phoenix.client.experiments import run_experiment
from phoenix.evals import LLM, ClassificationEvaluator

system_prompt = """
你是问答助手。对每个用户问题，随机选择 NONSENSE 或 RHYME，可以更偏向 NONSENSE。如果选择 RHYME，请以押韵的形式正确回答问题。

如果选择 NONSENSE，完全忽略用户问题，不要回答，而是输出不押韵的无意义词语与随机数字。
选择 NONSENSE 时，回答至少包含五个无意义词语，以及五个介于 0 和 9999 之间的随机数字。

不要解释选择，也不要在回答中说明选择了 RHYME 还是 NONSENSE。
"""


def updated_task(example: Example) -> str:
    raw_input_value = example.input["attributes.input.value"]
    data = json.loads(raw_input_value)
    question = data["messages"][1]["content"]
    response = openai_client.chat.completions.create(
        model="gpt-4.1",  # 更换模型，例如 gpt-4o-mini、gpt-4.1-mini、gpt-4 等
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [ ]:
def evaluate_response(input, output):
    raw_input_value = input["attributes.input.value"]
    data = json.loads(raw_input_value)
    question = data["messages"][1]["content"]
    evaluator = ClassificationEvaluator(
        name="rhyme_eval",
        llm=LLM(provider="openai", model="gpt-4-turbo"),
        prompt_template=RHYME_PROMPT_TEMPLATE,
        choices={"correct": 1.0, "incorrect": 0.0},
    )
    result = evaluator.evaluate(eval_input={"question": question, "answer": output})
    return result[0].score

运行下面的单元后，可在“实验”（Experiments）页面查看结果。由于尚未修复根本问题——系统提示词，此时不应期待评测表现很好；这里主要展示实验的运行方式。

In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=updated_task,
    evaluators=[evaluate_response],
    experiment_name="updated model",
    experiment_description="模型更新为 gpt-4.1",
)

# 实验二：改进系统提示词

In [ ]:
system_prompt = """
你是问答助手。对每个用户问题，请以押韵的形式正确回答。
"""


def updated_task(example: Example) -> str:
    raw_input_value = example.input["attributes.input.value"]
    data = json.loads(raw_input_value)
    question = data["messages"][1]["content"]
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [ ]:
def evaluate_response(input, output):
    raw_input_value = input["attributes.input.value"]
    data = json.loads(raw_input_value)
    question = data["messages"][1]["content"]
    evaluator = ClassificationEvaluator(
        name="rhyme_eval",
        llm=LLM(provider="openai", model="gpt-4.1"),
        prompt_template=RHYME_PROMPT_TEMPLATE,
        choices={"correct": 1.0, "incorrect": 0.0},
    )
    result = evaluator.evaluate(eval_input={"question": question, "answer": output})
    return result[0].score

此时预期能看到实验效果改善。优化系统提示词后，评估器应当标出明显更少的不合逻辑回答。

In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=updated_task,
    evaluators=[evaluate_response],
    experiment_name="更新系统提示词",
    experiment_description="更新系统提示词",
)

# 将修改应用到应用程序

完成一轮实验并在已标注追踪上确认效果后，可以更新应用，并在更广泛的数据集上测试。这有助于确认实验中的改进也适用于实际使用，并验证系统在更大规模下是否可靠。

In [ ]:
questions = [
    "What is the capital of France?",
    "Who wrote 'Pride and Prejudice'?",
    "What is the boiling point of water in Celsius?",
    "What is the largest planet in our solar system?",
    "Who developed the theory of relativity?",
    "What is the chemical symbol for gold?",
    "In which year did the Apollo 11 mission land on the moon?",
    "What language has the most native speakers worldwide?",
    "Which continent has the most countries?",
    "What is the square root of 144?",
    "What is the largest country in the world by land area?",
    "Why is the sky blue?",
    "Who painted the Mona Lisa?",
    "What is the smallest prime number?",
    "What gas do plants absorb from the atmosphere?",
    "Who was the first President of the United States?",
    "What is the currency of Japan?",
    "How many continents are there on Earth?",
    "What is the tallest mountain in the world?",
    "Who is the author of '1984'?",
]
questions_df = pd.DataFrame(questions, columns=["Questions"])
dataset = Client().datasets.create_dataset(
    dataframe=questions_df,
    input_keys=["Questions"],
    name="all-trivia-questions",
)

In [ ]:
system_prompt = """
你是问答助手。对每个用户问题，请以押韵的形式正确回答。
"""


# 遍历数据集并收集 Span
def complete_task(question) -> str:
    question_str = question["Questions"]
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question_str},
        ],
    )
    return response.choices[0].message.content


def evaluate_all_responses(input, output):
    evaluator = ClassificationEvaluator(
        name="rhyme_eval",
        llm=LLM(provider="openai", model="gpt-4o"),
        prompt_template=RHYME_PROMPT_TEMPLATE,
        choices={"correct": 1.0, "incorrect": 0.0},
    )
    result = evaluator.evaluate(eval_input={"question": input["Questions"], "answer": output})
    return result[0].score


experiment = run_experiment(
    dataset=dataset,
    task=complete_task,
    evaluators=[evaluate_all_responses],
    experiment_name="modified-system-prompt-full-dataset",
)

## 构建自定义标注界面的建议

下面的示例提示词可交给 [Lovable](https://lovable.dev)，用来开始构建自定义 LLM 追踪标注界面；请按需调整。你需要实现获取 Span 及向 Phoenix 发送标注的功能。下文也简要介绍了本示例的实现方式。

**给 Lovable 的提示词：**

构建一个用于标注 LLM Span 和 Trace 的平台：

1. 向用户收集地址、API 密钥和项目名，连接 Phoenix Cloud。
2. 从 Phoenix 加载追踪与 Span，可使用 [REST API](https://arize.com/docs/phoenix/sdk-api-reference/spans#get-v1-projects-project_identifier-spans) 或 [Python SDK](https://arize.com/docs/phoenix/tracing/how-to-tracing/feedback-and-annotations/evaluating-phoenix-traces#download-trace-dataset-from-phoenix)。
3. 按 `trace_id` 分组显示 Span，组间清晰分隔。
4. 允许标注者为单个 Span 或整条追踪指定标签、分数和解释。
5. 支持将标注写回 Phoenix，并重新加载查看更新。
6. 使用简洁、现代的界面设计。

**本标注界面的实现方式：**

✅ 前端（Lovable）：

- 使用 Lovable 方便地生成界面。
- 支持加载 LLM 追踪、按 `trace_id` 分组展示 Span，并填写标签、分数和解释。

✅ 后端（Render、FastAPI）：

- 使用 FastAPI，托管于 Render。
- 配置 CORS，让 Lovable 前端与后端通信。
- 使用两个主要端点：
  1. GET /v1/projects/{project_identifier}/spans
  2. POST /v1/span_annotations